# install packages

In [64]:
# %pip install concretedropout[jax]
# install dataset dependencies
# %pip install tensorflow tensorflow-datasets 

In [65]:
import tensorflow_datasets as tfds  # TFDS to download MNIST.
import tensorflow as tf  # TensorFlow / `tf.data` operations.
from tqdm.auto import tqdm

tf.random.set_seed(0)  # Set the random seed for reproducibility.

train_steps = 12_000
eval_every = 200
batch_size = 64

train_ds: tf.data.Dataset = tfds.load('mnist', split='train')
test_ds: tf.data.Dataset = tfds.load('mnist', split='test')

train_ds = train_ds.map(
  lambda sample: {
    'image': tf.cast(sample['image'], tf.float32) / 255,
    'label': sample['label'],
  }
)  # normalize train set
test_ds = test_ds.map(
  lambda sample: {
    'image': tf.cast(sample['image'], tf.float32) / 255,
    'label': sample['label'],
  }
)  # Normalize the test set.

# Create a shuffled dataset by allocating a buffer size of 1024 to randomly draw elements from.
train_ds = train_ds.repeat().shuffle(1024)
# Group into batches of `batch_size` and skip incomplete batches, prefetch the next sample to improve latency.
train_ds = train_ds.batch(batch_size, drop_remainder=True).take(train_steps).prefetch(1)
# Group into batches of `batch_size` and skip incomplete batches, prefetch the next sample to improve latency.
test_ds = test_ds.batch(batch_size, drop_remainder=True).prefetch(1)

In [66]:
import os 
os.environ["CONCRETEDROPOUT_BACKEND"] = "jax"
import sys
sys.path.append('../../src')
from  ConcreteDropout import ConcreteDropout, get_weight_regularizer, get_dropout_regularizer #TODO change

In [67]:
Ns = len(train_ds)
wr = get_weight_regularizer(Ns, l=1e-2, tau=1.0)
dr = get_dropout_regularizer(Ns, tau=1.0, cross_entropy_loss=True)
Ns, wr, dr

(12000, 8.333333333333334e-09, 8.333333333333333e-05)

In [68]:
from flax import nnx  # The Flax NNX API.
from functools import partial

class CNN(nnx.Module):
  """A simple CNN model."""

  def __init__(self, *, rngs: nnx.Rngs):
    self.conv1 = nnx.Conv(1, 32, kernel_size=(3, 3), rngs=rngs)

    self.conv2 = nnx.Conv(32, 64, kernel_size=(3, 3), rngs=rngs)
    self.conv2_cd = ConcreteDropout(self.conv2, broadcast_dims=(1,2),weight_regularizer=wr, dropout_regularizer=dr, init_min=0.1, init_max=0.2, temperature=0.3, rngs=rngs)

    self.avg_pool = partial(nnx.avg_pool, window_shape=(2, 2), strides=(2, 2))
    self.linear1 = nnx.Linear(3136, 1000, rngs=rngs)
    self.linear1_cd = ConcreteDropout(self.linear1, weight_regularizer=wr, dropout_regularizer=dr, init_min=0.4, init_max=0.5, temperature=0.3, rngs=rngs)
    self.linear2 = nnx.Linear(1000, 10, rngs=rngs)
    self.linear2_cd = ConcreteDropout(self.linear2, weight_regularizer=wr, dropout_regularizer=dr, init_min=0.4, init_max=0.5, temperature=0.3, rngs=rngs)

  def get_reg_loss(self):
    return (self.conv2_cd.reg_loss + self.linear1_cd.reg_loss + self.linear2_cd.reg_loss)[0]

  def __call__(self, x):
    x = self.avg_pool(nnx.relu(self.conv1(x)))
    
    x = self.avg_pool(nnx.relu(self.conv2_cd(x)))

    x = x.reshape(x.shape[0], -1)  # flatten
    x = nnx.relu(self.linear1_cd(x))
    x = self.linear2_cd(x)
    return x

# Instantiate the model.
model = CNN(rngs=nnx.Rngs(0))
# Visualize it.
# nnx.display(model)


In [69]:
import jax.numpy as jnp  # JAX NumPy

y = model(jnp.ones((1, 28, 28, 1)))
y

TypeError: dot_general requires contracting dimensions to have the same shape, got (4000,) and (1000,).

In [59]:
import optax

learning_rate = 0.005
momentum = 0.9

optimizer = nnx.Optimizer(model, optax.adamw(learning_rate, momentum))
metrics = nnx.MultiMetric(
  accuracy=nnx.metrics.Accuracy(),
  loss=nnx.metrics.Average('loss'),
)

In [60]:
def loss_fn(model: CNN, batch):
  logits = model(batch['image'])
  regularization_loss = model.get_reg_loss()
  loss = optax.softmax_cross_entropy_with_integer_labels(
    logits=logits, labels=batch['label']
  ).mean()
  loss += regularization_loss
  return loss, logits

@nnx.jit
def train_step(model: CNN, optimizer: nnx.Optimizer, metrics: nnx.MultiMetric, batch):
  """Train for a single step."""
  grad_fn = nnx.value_and_grad(loss_fn, has_aux=True)
  (loss, logits), grads = grad_fn(model, batch)
  metrics.update(loss=loss, logits=logits, labels=batch['label'])  # In-place updates.
  optimizer.update(grads)  # In-place updates.

@nnx.jit
def eval_step(model: CNN, metrics: nnx.MultiMetric, batch):
  loss, logits = loss_fn(model, batch)
  metrics.update(loss=loss, logits=logits, labels=batch['label'])  # In-place updates.

In [ ]:
metrics_history = {
  'train_loss': [],
  'train_accuracy': [],
  'test_loss': [],
  'test_accuracy': [],
}

pbar = tqdm(train_ds.as_numpy_iterator(), total=train_steps, desc="Training")

for step, batch in enumerate(pbar):
    # Run the optimization for one step and make a stateful update to the following:
    # - The train state's model parameters
    # - The optimizer state
    # - The training loss and accuracy batch metrics
    train_step(model, optimizer, metrics, batch)
    if step > 0 and (step % eval_every == 0 or step == train_steps - 1):  # One training epoch has passed.
        # Log the training metrics.
        for metric, value in metrics.compute().items():  # Compute the metrics.
            metrics_history[f'train_{metric}'].append(value)  # Record the metrics.
        metrics.reset()  # Reset the metrics for the test set.

        loss_v =  metrics_history['train_loss'][-1]
        pbar.set_postfix(loss=loss_v)
        

    # if step > 0 and (step % eval_every == 0 or step == train_steps - 1):  # One training epoch has passed.
    #   # Log the training metrics.
    #   for metric, value in metrics.compute().items():  # Compute the metrics.
    #     metrics_history[f'train_{metric}'].append(value)  # Record the metrics.
    #   metrics.reset()  # Reset the metrics for the test set.

    #   # Compute the metrics on the test set after each training epoch.
    #   for test_batch in test_ds.as_numpy_iterator():
    #     eval_step(model, metrics, test_batch)

    #   # Log the test metrics.
    #   for metric, value in metrics.compute().items():
    #     metrics_history[f'test_{metric}'].append(value)
    #   metrics.reset()  # Reset the metrics for the next training epoch.

    #   print(
    #     f"[train] step: {step}, "
    #     f"loss: {metrics_history['train_loss'][-1]}, "
    #     f"accuracy: {metrics_history['train_accuracy'][-1] * 100}"
    #   )
    #   print(
    #     f"[test] step: {step}, "
    #     f"loss: {metrics_history['test_loss'][-1]}, "
    #     f"accuracy: {metrics_history['test_accuracy'][-1] * 100}"
      # )

Training:   0%|          | 0/12000 [00:00<?, ?it/s]

In [63]:
nnx.sigmoid(model.conv2_cd.p_logit), nnx.sigmoid(model.linear1_cd.p_logit), nnx.sigmoid(model.linear2_cd.p_logit)

(Array([0.0066833], dtype=float32),
 Array([0.00355387], dtype=float32),
 Array([0.10127474], dtype=float32))